# 7장. RAG 시스템 평가 — 05. Heuristic 평가 (ROUGE, BLEU, METEOR, SemScore)

## 학습 목표

- **Heuristic(휴리스틱) 평가**의 개념과 장점 이해
- ROUGE, BLEU, METEOR, SemScore 네 가지 지표의 원리와 차이 파악
- LLM 호출 없이 빠르고 저렴하게 평가하는 방법 습득

## 사전 지식

- 04번 노트북에서 커스텀 평가자 작성 경험
- N-gram의 개념 (단어 연속열)

## Heuristic 평가란?

LLM 호출 없이 **통계적 알고리즘**으로 텍스트 품질을 측정하는 방법이에요.

| 특성 | Heuristic | LLM-as-Judge |
|------|-----------|--------------|
| **속도** | 밀리초 단위 | 1~3초 |
| **비용** | 없음 | API 비용 |
| **일관성** | 항상 동일 | 확률적 변동 |
| **정교함** | 제한적 | 높음 |
| **참조 답변** | 필수 | 선택적 |

> 🎯 **강의 포인트**: Heuristic 평가의 핵심 장점은 **결정론적(Deterministic)**이라는 점입니다. 같은 입력에 항상 같은 점수가 나옵니다. LLM-as-Judge는 프롬프트가 조금만 바뀌어도 점수가 달라질 수 있지만, ROUGE나 BLEU는 항상 동일한 결과를 보장합니다. CI/CD 파이프라인에 통합하기에 이상적입니다.

---

## 환경 설정

In [ ]:
# 필요한 패키지 설치
# !pip install -qU langsmith langchain langchain-openai rouge-score nltk


In [ ]:
from dotenv import load_dotenv
import os

# 환경변수 로드
load_dotenv()

# macOS에서 FAISS 사용 시 OpenMP 중복 로드 방지
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# LangSmith 프로젝트 설정
os.environ["LANGSMITH_PROJECT"] = "05-Eval-Heuristic"

# LangSmith 설정 확인
print(f"LANGCHAIN_API_KEY: {'설정됨' if os.getenv('LANGCHAIN_API_KEY') else '미설정'}")
print(f"LANGSMITH_PROJECT: {os.getenv('LANGSMITH_PROJECT')}")

# ---------------------------------------------------
# [LangSmith UI 확인 방법]
# 1. https://smith.langchain.com 접속
# 2. 좌측 Projects 클릭
# 3. "Eval-Heuristic" 프로젝트 선택
# 4. 주안점: Experiments 탭 → 휴리스틱 평가 점수 (BLEU, ROUGE 등)
# ---------------------------------------------------

---

## RAG 시스템 구축

앞 노트북과 동일한 RAG 시스템을 구축해요.

In [ ]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ---------------------------------------------------
# 트랜스포머 관련 한국어 컨텍스트 문서
# ---------------------------------------------------
documents = [
    Document(page_content=(
        "트랜스포머(Transformer)는 2017년 구글이 'Attention Is All You Need' 논문에서 "
        "제안한 신경망 아키텍처입니다. 기존의 순환 신경망(RNN)이나 합성곱 신경망(CNN) 없이 "
        "어텐션 메커니즘만으로 시퀀스를 처리합니다. 인코더와 디코더 각각 6개의 동일한 레이어를 "
        "쌓아 올린 구조이며, 모든 레이어의 출력 차원은 d_model = 512입니다."
    )),
    Document(page_content=(
        "셀프 어텐션(Self-Attention)은 시퀀스 내 모든 위치 간의 관계를 계산하는 메커니즘입니다. "
        "입력에서 쿼리(Query), 키(Key), 값(Value) 벡터를 생성하고, 쿼리와 키의 유사도로 "
        "어텐션 가중치를 계산합니다. 이를 통해 모델이 입력의 어떤 부분에 집중해야 하는지 학습합니다."
    )),
    Document(page_content=(
        "멀티헤드 어텐션(Multi-Head Attention)은 셀프 어텐션을 여러 개의 '헤드'로 병렬 수행하는 "
        "기법입니다. 각 헤드는 서로 다른 표현 부분공간(representation subspace)에서 정보를 추출합니다. "
        "논문에서는 8개의 헤드를 사용했으며, 각 헤드의 출력을 연결한 뒤 선형 변환을 적용합니다."
    )),
    Document(page_content=(
        "포지셔널 인코딩(Positional Encoding)은 트랜스포머에 단어의 위치 정보를 제공합니다. "
        "트랜스포머는 순환 구조가 없어 단어의 순서를 알 수 없으므로, 사인(sin)과 코사인(cos) "
        "함수 기반의 위치 벡터를 입력 임베딩에 더해줍니다."
    )),
    Document(page_content=(
        "트랜스포머의 인코더는 입력 시퀀스를 연속적인 표현으로 변환하고, 디코더는 이를 바탕으로 "
        "출력을 한 토큰씩 생성합니다. 디코더에는 마스크드 셀프 어텐션이 추가되어 미래 토큰 참조를 "
        "방지합니다. 인코더-디코더 어텐션으로 디코더가 입력의 관련 부분에 집중합니다."
    )),
    Document(page_content=(
        "트랜스포머는 RNN과 달리 시퀀스를 병렬로 처리할 수 있어 훈련 속도가 크게 빠릅니다. "
        "RNN은 순차 처리가 필수지만 트랜스포머는 모든 위치를 동시에 처리합니다. "
        "장거리 의존성도 더 효과적으로 포착할 수 있습니다."
    )),
    Document(page_content=(
        "스케일드 닷 프로덕트 어텐션은 쿼리와 키의 내적을 키 차원의 제곱근으로 나누어 스케일링합니다. "
        "스케일링 없이는 내적 값이 커져 소프트맥스의 기울기가 매우 작아집니다. "
        "스케일링 후 소프트맥스를 적용하여 가중치를 구하고, 값 벡터에 곱하여 출력을 얻습니다."
    )),
]

# 벡터 DB 생성
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever()

# 3단계: 프롬프트 정의
prompt = PromptTemplate.from_template(
    """주어진 컨텍스트를 바탕으로 질문에 답변하세요. 간결하고 구체적으로 답변하세요.

컨텍스트: {context}

질문: {question}

답변:"""
)

# 4단계: LLM 및 체인 생성
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 평가용 함수: LangSmith evaluate()에 전달하는 래퍼 함수
def rag_answer_function(inputs: dict) -> dict:
    """RAG 시스템 답변 함수"""
    question = inputs["question"]
    answer = rag_chain.invoke(question)
    return {"answer": answer}

print("RAG 시스템 구축 완료")

> 🔑 **핵심 개념**: ROUGE는 **Recall** 중심, BLEU는 **Precision** 중심입니다. 요약 평가는 참조 답변의 핵심 정보를 얼마나 포함하는지(Recall)가 중요해서 ROUGE를 씁니다. 번역은 생성된 텍스트가 얼마나 정확한지(Precision)가 중요해서 BLEU를 씁니다. RAG QA는 ROUGE-L이 가장 적합합니다.

---

## 1. ROUGE (Recall-Oriented Understudy for Gisting Evaluation)

**자동 요약 평가**에 가장 많이 사용되는 지표예요. 생성 텍스트와 참조 텍스트의 N-gram 중복을 측정하며, **Recall(재현율)** 중심으로 설계되었어요.

### ROUGE 종류

| 종류 | 측정 단위 | 특징 |
|------|---------|------|
| **ROUGE-1** | 단어(unigram) | 개별 단어 일치 |
| **ROUGE-2** | 2-gram | 연속 2단어 일치 |
| **ROUGE-L** | 최장 공통 부분 수열(LCS) | 순서를 고려한 일치 |

> **실무 팁**: RAG 시스템 평가에는 ROUGE-L이 가장 적합해요. 단어 순서도 고려하면서 유연한 비교가 가능합니다.

> ⚠️ **자주 하는 실수**: BLEU는 짧은 답변에 **Brevity Penalty**를 적용해 점수가 급격히 낮아집니다. RAG 답변은 길이가 일정하지 않으므로 BLEU만으로 평가하면 왜곡이 생깁니다. BLEU는 단독으로 쓰지 말고 ROUGE-L이나 SemScore와 함께 사용하세요.

In [ ]:
# ---------------------------------------------------
# ROUGE 평가자 구현 (factory 패턴으로 metric 선택 가능)
# ---------------------------------------------------
# ============================================================
# TODO: rouge_evaluator(metric="rougeL") 팩토리 함수를 완성하세요
# 힌트: RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)로 점수 계산 후 fmeasure 반환
# 예상 결과: {"key": "ROUGE", "score": 0~1}
# ============================================================

from langsmith.schemas import Run, Example
from rouge_score import rouge_scorer

def rouge_evaluator(metric: str = "rougeL") -> callable:
    """
    ROUGE 평가자 생성 함수
    
    Args:
        metric: 'rouge1', 'rouge2', 'rougeL' 중 선택
        
    Returns:
        평가자 함수
    """
    def _rouge_evaluator(run: Run, example: Example) -> dict:
        # 생성 답변과 참조 답변 가져오기
        prediction = run.outputs.get("answer", "")
        reference = example.outputs.get("answer", "")
        
        if not reference or not prediction:
            return {"key": "ROUGE", "score": 0.0}
        
        # ROUGE 점수 계산
        scorer = rouge_scorer.RougeScorer(
            ["rouge1", "rouge2", "rougeL"], 
            use_stemmer=True  # 어간 추출로 유사 단어 매칭
        )
        scores = scorer.score(reference, prediction)
        
        # F-measure: Precision과 Recall의 조화 평균
        rouge_score = scores[metric].fmeasure
        
        return {"key": "ROUGE", "score": rouge_score}
    
    return _rouge_evaluator

# 간단한 테스트
print("=" * 60)
print("ROUGE 테스트")
print("=" * 60)

reference = "트랜스포머는 어텐션 메커니즘에만 의존하는 신경망 아키텍처입니다."
prediction = "트랜스포머는 어텐션 메커니즘을 기반으로 한 신경망 구조입니다."

scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
score = scorer.score(reference, prediction)
print(f"\n참조: {reference}")
print(f"예측: {prediction}")
print(f"ROUGE-L F1: {score['rougeL'].fmeasure:.4f}")

---

## 2. BLEU (Bilingual Evaluation Understudy)

원래 **기계 번역 평가**를 위해 개발된 지표로, **Precision(정밀도)** 중심이에요. 1~4-gram 정밀도의 기하평균으로 계산됩니다.

> **주의**: BLEU는 짧은 답변에 불리하고, 단어 순서보다 단어 선택에 치중해요. 번역이 아닌 QA 태스크에서는 ROUGE-L이 더 적합한 경우가 많습니다.

In [ ]:
# ---------------------------------------------------
# BLEU 평가자 구현
# ---------------------------------------------------
# ============================================================
# TODO: sentence_bleu([reference_tokens], prediction_tokens)를 사용하는 평가자를 완성하세요
# 힌트: 공백으로 split()하여 토큰화하고, ZeroDivisionError 예외 처리 필요
# 예상 결과: {"key": "BLEU", "score": 0~1}
# ============================================================

from nltk.translate.bleu_score import sentence_bleu
import warnings
warnings.filterwarnings('ignore')

def bleu_evaluator(run: Run, example: Example) -> dict:
    """
    BLEU 평가자
    
    BLEU는 1-gram ~ 4-gram의 정밀도를 기하평균하여 계산
    """
    prediction = run.outputs.get("answer", "")
    reference = example.outputs.get("answer", "")
    
    if not reference or not prediction:
        return {"key": "BLEU", "score": 0.0}
    
    # 단어로 토큰화
    reference_tokens = reference.split()
    prediction_tokens = prediction.split()
    
    # BLEU 점수 계산
    try:
        # sentence_bleu: 참조는 리스트의 리스트 형식 (여러 참조 답변 지원)
        bleu_score = sentence_bleu([reference_tokens], prediction_tokens)
    # ValueError/ZeroDivisionError: 토큰이 비어있거나 n-gram 계산 실패 시 처리
    except (ValueError, ZeroDivisionError) as e:
        print(f"BLEU 계산 오류: {e}")
        bleu_score = 0.0
    
    return {"key": "BLEU", "score": bleu_score}

# 간단한 테스트
print("\n" + "=" * 60)
print("BLEU 테스트")
print("=" * 60)

reference = "트랜스포머는 셀프 어텐션 메커니즘을 사용합니다"
prediction = "트랜스포머는 셀프 어텐션을 활용합니다"

ref_tokens = reference.split()
pred_tokens = prediction.split()
bleu = sentence_bleu([ref_tokens], pred_tokens)

print(f"\n참조: {reference}")
print(f"예측: {prediction}")
print(f"BLEU (split 토큰화): {bleu:.4f}")
print("→ 한국어에서 split()은 조사가 붙은 어절 단위로 분리하므로 정확도가 낮아요")

> 💡 **실무 팁**: 위 코드에서 `split()`은 공백 기준 토큰화로, 한국어에서는 조사가 붙은 **어절 단위**로 분리됩니다. 예를 들어 "트랜스포머를"과 "트랜스포머는"은 같은 단어지만 `split()`으로는 완전히 다른 토큰으로 처리돼요. 이 문제를 해결하려면 **형태소 분석기**(Kiwi 등)를 사용해야 합니다. 이 노트북 뒷부분에서 `split()` vs Kiwi 비교를 직접 확인해볼게요.

> 🔑 **핵심 개념**: SemScore는 N-gram 기반 지표들이 놓치는 **의미적 동치**를 포착합니다. "빠르다"와 "신속하다"는 ROUGE에서 0점이지만 SemScore에서는 높은 유사도를 가집니다. 한국어처럼 어휘 변이가 많은 언어에서 특히 유용합니다.

---

## 3. METEOR (Metric for Evaluation of Translation with Explicit ORdering)

BLEU보다 **유연한 평가**를 제공해요. 어간(stem)과 동의어(synonym)를 함께 고려해서 표면적으로 다른 단어도 의미가 같으면 일치로 처리합니다.

**예시**: "quick"와 "fast"는 BLEU에서 불일치지만 METEOR에서는 일치

In [ ]:
# ---------------------------------------------------
# METEOR 평가자 구현 (동의어, 어간 고려)
# ---------------------------------------------------
# ============================================================
# TODO: meteor_score.meteor_score([reference_tokens], prediction_tokens)를 사용하는 평가자를 완성하세요
# 힌트: WordNet 리소스를 먼저 다운로드하고 split()으로 토큰화
# 예상 결과: {"key": "METEOR", "score": 0~1}
# ============================================================

from nltk.translate import meteor_score
from nltk.corpus import wordnet as wn
import nltk

# WordNet 다운로드 (최초 1회) — 동의어 판단에 사용
try:
    wn.ensure_loaded()
# LookupError: NLTK 리소스가 없을 때만 다운로드합니다.
# bare except를 쓰면 KeyboardInterrupt 등 예상치 못한 오류까지 삼켜버립니다.
except LookupError:
    nltk.download('wordnet', quiet=True)
    nltk.download('omw-1.4', quiet=True)
    wn.ensure_loaded()

def meteor_evaluator(run: Run, example: Example) -> dict:
    """
    METEOR 평가자
    
    동의어와 어간을 고려하여 BLEU보다 유연한 평가
    """
    prediction = run.outputs.get("answer", "")
    reference = example.outputs.get("answer", "")
    
    if not reference or not prediction:
        return {"key": "METEOR", "score": 0.0}
    
    # 토큰화
    reference_tokens = reference.split()
    prediction_tokens = prediction.split()
    
    # METEOR 점수 계산
    try:
        meteor = meteor_score.meteor_score([reference_tokens], prediction_tokens)
    # ValueError: 토큰화 또는 점수 계산 중 잘못된 입력 처리
    except (ValueError, ZeroDivisionError) as e:
        print(f"METEOR 계산 오류: {e}")
        meteor = 0.0
    
    return {"key": "METEOR", "score": meteor}

# 간단한 테스트
print("\n" + "=" * 60)
print("METEOR 테스트")
print("=" * 60)

reference = "트랜스포머는 시퀀스를 병렬로 처리할 수 있는 아키텍처입니다"
prediction = "트랜스포머는 병렬 처리가 가능한 모델 아키텍처입니다"

ref_tokens = reference.split()
pred_tokens = prediction.split()
meteor = meteor_score.meteor_score([ref_tokens], pred_tokens)

print(f"\n참조: {reference}")
print(f"예측: {prediction}")
print(f"METEOR: {meteor:.4f}")
print("→ METEOR의 동의어 매칭(WordNet)은 영어 전용이라 한국어에서는 효과가 제한적이에요")

---

## 4. SemScore (Semantic Similarity Score)

**문장 임베딩(Sentence Embedding)** 기반 의미 유사도 측정이에요. N-gram 기반 지표들과 달리 단어가 달라도 의미가 같으면 높은 점수를 줍니다.

**차이 예시**:

```
참조: "트랜스포머는 어텐션 메커니즘을 사용합니다"
예측: "트랜스포머는 주의 집중 기법을 활용합니다"

ROUGE-1: 낮음 (단어 일치 적음)
SemScore: 높음 (의미가 동일)
```

In [ ]:
# ---------------------------------------------------
# SemScore 평가자 구현 (OpenAI 임베딩 기반)
# ---------------------------------------------------
# ============================================================
# TODO: OpenAIEmbeddings를 사용해 의미 유사도 평가자를 완성하세요
# 힌트: embeddings.embed_query(text)로 벡터를 생성하고, 코사인 유사도를 계산
# 예상 결과: {"key": "sem_score", "score": 0~1}
# ============================================================

import numpy as np
from langchain_openai import OpenAIEmbeddings

# OpenAI 임베딩 모델 초기화
sem_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

def cosine_similarity(a: list[float], b: list[float]) -> float:
    """두 벡터 간 코사인 유사도를 계산해요"""
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def semscore_evaluator(run: Run, example: Example) -> dict:
    """
    SemScore 평가자
    
    OpenAI 임베딩 기반 의미 유사도로 N-gram보다 의미를 잘 포착
    """
    prediction = run.outputs.get("answer", "")
    reference = example.outputs.get("answer", "")
    
    if not reference or not prediction:
        return {"key": "sem_score", "score": 0.0}
    
    # 임베딩 생성
    pred_embedding = sem_embeddings.embed_query(prediction)
    ref_embedding = sem_embeddings.embed_query(reference)
    
    # 코사인 유사도 계산 (텍스트 임베딩은 거의 항상 [0, 1] 범위)
    similarity = cosine_similarity(pred_embedding, ref_embedding)
    normalized_score = max(0.0, min(1.0, similarity))
    
    return {"key": "sem_score", "score": normalized_score}

# 간단한 테스트
print("\n" + "=" * 60)
print("SemScore 테스트")
print("=" * 60)

reference = "트랜스포머는 어텐션 메커니즘을 사용합니다"
prediction = "트랜스포머는 주의 집중 기법을 활용합니다"

ref_emb = sem_embeddings.embed_query(reference)
pred_emb = sem_embeddings.embed_query(prediction)
similarity = cosine_similarity(ref_emb, pred_emb)

print(f"\n참조: {reference}")
print(f"예측: {prediction}")
print(f"SemScore: {similarity:.4f}")
print("→ '어텐션 메커니즘'과 '주의 집중 기법'은 같은 의미 — SemScore가 이를 포착합니다")

> 💡 **실무 팁**: 코사인 유사도의 이론적 범위는 [-1, 1]이지만, 텍스트 임베딩(Sentence-Transformers, OpenAI 등)에서는 거의 항상 **[0, 1]** 범위의 값이 나옵니다. 인터넷에서 자주 보이는 `(similarity + 1) / 2` 정규화는 0.85 점수를 0.925로 부풀리는 부작용이 있어요. 텍스트 도메인에서는 `max(0, similarity)` 클리핑이 더 직관적입니다.

---

## 5. 한국어 토큰화: split() vs Kiwi 형태소 분석

한국어 텍스트에서 `split()`은 **어절 단위**(공백 기준)로 분리합니다. 하지만 한국어는 조사가 단어에 붙기 때문에, 같은 단어라도 조사에 따라 완전히 다른 토큰으로 처리돼요.

```
"트랜스포머는" ≠ "트랜스포머가" ≠ "트랜스포머를"  ← split() 기준으로 모두 다른 토큰
"트랜스포머"  = "트랜스포머"  = "트랜스포머"   ← Kiwi 형태소 분석 후 동일 토큰
```

**Kiwi(키위)**는 한국어 형태소 분석기로, 어절을 형태소 단위로 분리합니다. 이를 통해 조사/어미 변화에 관계없이 핵심 단어를 추출할 수 있어요.

> 🎯 **강의 포인트**: 토큰화 방식이 평가 점수를 극적으로 바꿀 수 있습니다. 아래에서 같은 텍스트 쌍에 대해 `split()` vs Kiwi 토큰화로 BLEU/ROUGE 점수가 어떻게 달라지는지 직접 확인해보세요.

In [ ]:
# ---------------------------------------------------
# split() vs Kiwi 토큰화 비교
# ---------------------------------------------------
from kiwipiepy import Kiwi
from nltk.translate.bleu_score import sentence_bleu
from rouge_score import rouge_scorer

kiwi = Kiwi()

def kiwi_tokenize(text: str) -> list[str]:
    """Kiwi 형태소 분석기로 토큰화"""
    return [token.form for token in kiwi.tokenize(text)]

# 테스트 문장 — 조사만 다른 경우
reference = "트랜스포머는 어텐션 메커니즘에만 의존하는 신경망 아키텍처입니다"
prediction = "트랜스포머가 어텐션 메커니즘을 기반으로 하는 신경망 아키텍처이다"

# split() 토큰화
ref_split = reference.split()
pred_split = prediction.split()

# Kiwi 토큰화
ref_kiwi = kiwi_tokenize(reference)
pred_kiwi = kiwi_tokenize(prediction)

print("=" * 70)
print("토큰화 방식 비교")
print("=" * 70)
print(f"\n참조: {reference}")
print(f"예측: {prediction}")
print(f"\n[split() 토큰화]")
print(f"  참조: {ref_split}")
print(f"  예측: {pred_split}")
print(f"\n[Kiwi 형태소 분석]")
print(f"  참조: {ref_kiwi}")
print(f"  예측: {pred_kiwi}")

# BLEU 비교
bleu_split = sentence_bleu([ref_split], pred_split)
bleu_kiwi = sentence_bleu([ref_kiwi], pred_kiwi)

# ROUGE-L 비교
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
rouge_split = scorer.score(reference, prediction)["rougeL"].fmeasure

# Kiwi 토큰을 공백으로 합쳐서 ROUGE 계산 (ROUGE는 내부적으로 공백 분리)
ref_kiwi_str = " ".join(ref_kiwi)
pred_kiwi_str = " ".join(pred_kiwi)
rouge_kiwi = scorer.score(ref_kiwi_str, pred_kiwi_str)["rougeL"].fmeasure

print(f"\n{'지표':<12} {'split()':>10} {'Kiwi':>10} {'차이':>10}")
print("-" * 45)
print(f"{'BLEU':<12} {bleu_split:>10.4f} {bleu_kiwi:>10.4f} {bleu_kiwi - bleu_split:>+10.4f}")
print(f"{'ROUGE-L':<12} {rouge_split:>10.4f} {rouge_kiwi:>10.4f} {rouge_kiwi - rouge_split:>+10.4f}")
print("\n→ Kiwi가 조사를 분리해주므로 핵심 단어 일치율이 높아져 점수가 올라갑니다")


### 시나리오별 평가 지표 비교

같은 참조 답변에 대해 세 가지 시나리오를 비교해볼게요:

1. **올바른 답변 (다른 표현)**: 의미는 같지만 단어를 바꿔 표현
2. **틀린 정보**: 사실과 다른 내용을 포함
3. **단어 순서만 다름**: 핵심 단어는 같지만 순서를 뒤바꿈

> ⚠️ **주의**: N-gram 지표(ROUGE, BLEU)는 **의미를 이해하지 못합니다**. 틀린 정보라도 단어가 겹치면 높은 점수가 나오고, 올바른 답변이라도 단어가 다르면 낮은 점수가 나올 수 있어요. 이것이 SemScore를 함께 사용해야 하는 이유입니다.

In [ ]:
# ---------------------------------------------------
# 시나리오별 평가 지표 비교 (Kiwi 토큰화 적용)
# ---------------------------------------------------
reference = "트랜스포머는 순환이나 합성곱 없이 어텐션 메커니즘에만 의존하는 신경망 아키텍처입니다"

scenarios = {
    "올바른 답변 (다른 표현)": "트랜스포머는 어텐션 기법만을 활용하며 순환 구조나 합성곱을 사용하지 않는 신경망 모델입니다",
    "틀린 정보": "트랜스포머는 합성곱 신경망(CNN)을 기반으로 하며 순환 구조를 핵심으로 사용하는 아키텍처입니다",
    "단어 순서만 다름": "신경망 아키텍처입니다 어텐션 메커니즘에만 의존하는 순환이나 합성곱 없이 트랜스포머는",
}

print("=" * 70)
print(f"참조: {reference}")
print("=" * 70)

scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

for name, pred in scenarios.items():
    # Kiwi 토큰화
    ref_tokens = kiwi_tokenize(reference)
    pred_tokens = kiwi_tokenize(pred)
    
    # BLEU (Kiwi)
    bleu = sentence_bleu([ref_tokens], pred_tokens)
    
    # ROUGE-L (Kiwi 토큰을 공백 결합)
    rouge = scorer.score(" ".join(ref_tokens), " ".join(pred_tokens))["rougeL"].fmeasure
    
    # SemScore
    ref_emb = sem_embeddings.embed_query(reference)
    pred_emb = sem_embeddings.embed_query(pred)
    semscore = cosine_similarity(ref_emb, pred_emb)
    
    print(f"\n[{name}]")
    print(f"  예측: {pred}")
    print(f"  BLEU: {bleu:.4f} | ROUGE-L: {rouge:.4f} | SemScore: {semscore:.4f}")

print("\n" + "=" * 70)
print("분석:")
print("  - '틀린 정보'도 ROUGE가 높을 수 있음 → 단어가 겹치면 의미와 무관하게 점수 상승")
print("  - '올바른 답변(다른 표현)'은 ROUGE 낮고 SemScore 높음 → 의미 평가의 필요성")
print("  - '단어 순서만 다름'은 ROUGE-1은 높지만 ROUGE-L은 낮음 → LCS가 순서 고려")


---

## LangSmith 데이터셋 준비

---

## 평가 결과 해석

4가지 Heuristic 평가자의 결과를 질문별로 비교해볼게요. 실제 실행 결과는 다음과 유사합니다:

| 질문 | ROUGE-L | BLEU | METEOR | SemScore |
|------|:-------:|:----:|:------:|:--------:|
| 트랜스포머 아키텍처란? | 0.000 | 0.000 | 0.085 | 0.506 |
| 셀프 어텐션은 어떻게? | 0.000 | 0.201 | 0.421 | 0.799 |
| 트랜스포머의 장점은? | 0.000 | 0.133 | 0.275 | 0.742 |
| 멀티헤드 어텐션이란? | 0.000 | 0.000 | 0.177 | 0.662 |
| 포지셔널 인코딩의 역할은? | 0.000 | 0.280 | 0.452 | 0.774 |

### 결과 해석

**1. ROUGE-L이 전부 0.000인 이유**
- ROUGE는 내부적으로 **공백 기준 토큰화**를 사용합니다. 한국어에서는 조사가 붙은 어절 단위로 분리되므로, "메커니즘에만"과 "메커니즘만으로"가 완전히 다른 토큰으로 처리됩니다.
- → 앞서 다룬 **Kiwi 형태소 분석기**로 토큰화하면 이 문제가 해결됩니다.

**2. BLEU도 매우 낮음 (0.0~0.28)**
- 같은 이유로 어절 단위 N-gram 정밀도가 낮게 계산됩니다.
- 0.280으로 가장 높은 "포지셔널 인코딩"은 참조 답변과 공유하는 어절이 상대적으로 많았기 때문입니다.

**3. SemScore만 의미 있는 점수 (0.5~0.8)**
- 임베딩 기반이라 토큰화 문제에 영향 받지 않습니다.
- "셀프 어텐션"(0.799)은 참조와 의미가 매우 유사하고, "트랜스포머 아키텍처"(0.506)는 RAG가 훨씬 상세한 답변을 생성해서 벡터 거리가 벌어진 것입니다.

**4. 핵심 교훈**
- 한국어 텍스트에서 ROUGE/BLEU를 `split()` 기반으로 쓰면 **사실상 무의미한 점수**가 나옵니다.
- 한국어 평가에서는 **Kiwi 토큰화 + SemScore 조합**이 가장 신뢰할 수 있습니다.

In [ ]:
# ---------------------------------------------------
# 질문별 Heuristic 평가 결과 상세 분석
# ---------------------------------------------------
import pandas as pd

rows = []
for result in experiment_results:
    question = result["example"].inputs["question"]
    reference = result["example"].outputs.get("answer", "")[:30] + "..."
    
    scores = {}
    for eval_result in result["evaluation_results"]["results"]:
        scores[eval_result.key] = eval_result.score
    
    rows.append({
        "질문": question[:18] + "...",
        "ROUGE-L": f"{scores.get('ROUGE', 0):.3f}",
        "BLEU": f"{scores.get('BLEU', 0):.3f}",
        "METEOR": f"{scores.get('METEOR', 0):.3f}",
        "SemScore": f"{scores.get('sem_score', 0):.3f}",
    })

df_result = pd.DataFrame(rows)
print("=" * 75)
print("Heuristic 평가 결과 비교 (질문별)")
print("=" * 75)
print(df_result.to_string(index=False))

# 평균 점수
print("\n" + "-" * 75)
print("평균 점수:")
for col in ["ROUGE-L", "BLEU", "METEOR", "SemScore"]:
    avg = sum(float(r[col]) for r in rows) / len(rows)
    print(f"  {col:10s}: {avg:.3f}")

print("\n해석:")
print("  - BLEU가 전반적으로 낮다면 → RAG 답변이 참조 답변보다 길거나 표현이 다른 것")
print("  - ROUGE-L > BLEU라면 → 핵심 단어는 포함하지만 N-gram 정밀도가 낮은 것")
print("  - SemScore가 높고 ROUGE가 낮다면 → 표현은 다르지만 의미는 올바른 좋은 답변")


---

## 정리

### Heuristic 지표 비교

| 지표 | 측정 방식 | 장점 | 단점 | 적합한 태스크 |
|------|---------|------|------|-------------|
| **ROUGE** | N-gram 중복 (Recall) | 요약 평가 표준 | 표면적 일치만 측정 | 자동 요약 |
| **BLEU** | N-gram 정밀도 | 번역 평가 표준 | 짧은 문장에 불리 | 기계 번역 |
| **METEOR** | 동의어+어간+순서 | 유연한 평가 | 영어 동의어만 지원 | 번역, 요약 |
| **SemScore** | 임베딩 코사인 유사도 | 의미 포착 우수 | 참조 답변 필수 | QA, 요약 |

### N-gram 지표의 한계와 한국어 토큰화

N-gram 지표(ROUGE, BLEU, METEOR)는 표면적 단어 일치에 의존해요.

- **틀린 정보라도 단어가 겹치면 점수가 높게** 나올 수 있음
- **올바른 답변이라도 표현이 다르면 점수가 낮게** 나올 수 있음
- 한국어에서는 `split()` 대신 **Kiwi 형태소 분석기**로 토큰화해야 조사/어미에 의한 점수 왜곡을 방지할 수 있음

→ **SemScore와 함께 사용하는 것**이 좋아요.

### 지표 선택 가이드

- **요약 품질 체크**: ROUGE (+ Kiwi 토큰화)
- **번역 품질 체크**: BLEU 또는 METEOR
- **의미 유사도가 중요할 때**: SemScore
- **종합 평가**: 네 가지 모두 + LLM-as-Judge

### 다음 노트북 예고

다음에는 RAG 시스템에서 가장 중요한 **Groundedness(근거성) 평가** — 답변이 컨텍스트에만 근거하는지, 즉 할루시네이션이 있는지 — 를 집중적으로 다뤄볼게요.

In [ ]:
from langsmith import Client
from langsmith import utils as ls_utils

client = Client()
dataset_name = "transformer-qa-heuristic-eval"

# 평가용 데이터 (질문 + 참조 답변)
qa_pairs = [
    {
        "question": "트랜스포머 아키텍처란 무엇인가요?",
        "answer": "트랜스포머는 순환이나 합성곱 없이 어텐션 메커니즘에만 전적으로 의존하는 신경망 아키텍처입니다."
    },
    {
        "question": "셀프 어텐션은 어떻게 작동하나요?",
        "answer": "셀프 어텐션은 시퀀스 내 모든 위치 간의 어텐션 가중치를 계산하여, 모델이 입력의 서로 다른 부분의 중요도를 반영할 수 있게 합니다."
    },
    {
        "question": "트랜스포머의 장점은 무엇인가요?",
        "answer": "트랜스포머는 시퀀스를 병렬로 처리할 수 있고, 장거리 의존성을 더 효과적으로 포착하며, RNN보다 훈련 효율이 높습니다."
    },
    {
        "question": "멀티헤드 어텐션이란 무엇인가요?",
        "answer": "멀티헤드 어텐션은 여러 개의 어텐션 함수를 병렬로 수행하여, 모델이 서로 다른 위치의 서로 다른 표현 부분공간 정보를 동시에 참조할 수 있게 합니다."
    },
    {
        "question": "포지셔널 인코딩의 역할은 무엇인가요?",
        "answer": "포지셔널 인코딩은 트랜스포머에 단어의 위치 정보를 제공하여, 순환 구조 없이도 시퀀스의 순서를 파악할 수 있게 합니다."
    },
]

# 데이터셋 생성 또는 사용
try:
    dataset = client.read_dataset(dataset_name=dataset_name)
    print(f"✅ 기존 데이터셋 사용: {dataset_name}")
except ls_utils.LangSmithNotFoundError:
    dataset = client.create_dataset(
        dataset_name=dataset_name,
        description="트랜스포머 Q&A 휴리스틱 평가용"
    )
    
    for qa in qa_pairs:
        client.create_example(
            inputs={"question": qa["question"]},
            outputs={"answer": qa["answer"]},
            dataset_id=dataset.id
        )
    print(f"✅ 새 데이터셋 생성: {dataset_name} ({len(qa_pairs)}개 예제)")

---

## 4가지 Heuristic 평가 실행

네 가지 평가자를 한꺼번에 실행해서 결과를 비교해볼게요.

In [ ]:
# ---------------------------------------------------
# 4가지 Heuristic 평가자 조합하여 종합 평가 실행
# ---------------------------------------------------
# ============================================================
# TODO: evaluate()에 4가지 heuristic 평가자를 전달하여 평가를 실행하세요
# 힌트: evaluators=[rouge_evaluator(metric="rougeL"), bleu_evaluator, meteor_evaluator, semscore_evaluator]
# 예상 결과: experiment_results.experiment_name 출력
# ============================================================

from langsmith.evaluation import evaluate

# 네 가지 Heuristic 평가자 리스트
heuristic_evaluators = [
    rouge_evaluator(metric="rougeL"),  # ROUGE-L: 최장 공통 부분 수열 기반
    bleu_evaluator,                     # BLEU: N-gram 정밀도 기반
    meteor_evaluator,                   # METEOR: 동의어+어간 고려
    semscore_evaluator,                 # SemScore: 임베딩 의미 유사도
]

# 평가 실행
experiment_results = evaluate(
    rag_answer_function,
    data=dataset_name,
    evaluators=heuristic_evaluators,
    experiment_prefix="heuristic-eval",
    metadata={
        "model": "gpt-4o-mini",
        "evaluator_types": "ROUGE, BLEU, METEOR, SemScore"
    }
)

print("\n" + "=" * 60)
print("Heuristic 평가 완료!")
print("=" * 60)
print(f"\n결과 실험 이름: {experiment_results.experiment_name}")
print("\nLangSmith 대시보드에서 다음 메트릭을 확인할 수 있습니다:")
print("  - ROUGE: N-gram 중복 기반 (요약 평가)")
print("  - BLEU: N-gram 정밀도 기반 (번역 평가)")
print("  - METEOR: 동의어+어간 고려 (유연한 평가)")
print("  - sem_score: 임베딩 기반 의미 유사도")

---

## 정리

### Heuristic 지표 비교

| 지표 | 측정 방식 | 장점 | 단점 | 적합한 태스크 |
|------|---------|------|------|-------------|
| **ROUGE** | N-gram 중복 (Recall) | 요약 평가 표준 | 표면적 일치만 측정 | 자동 요약 |
| **BLEU** | N-gram 정밀도 | 번역 평가 표준 | 짧은 문장에 불리 | 기계 번역 |
| **METEOR** | 동의어+어간+순서 | 유연한 평가 | 계산 복잡도 높음 | 번역, 요약 |
| **SemScore** | 임베딩 코사인 유사도 | 의미 포착 우수 | 참조 답변 필수 | QA, 요약 |

### N-gram 지표의 한계

N-gram 지표(ROUGE, BLEU, METEOR)는 표면적 단어 일치에 의존해요. 단어를 바꿔도 의미가 같은 경우를 놓칩니다. RAG 시스템의 답변은 표현 방식이 다양할 수 있으므로 **SemScore와 함께 사용하는 것**이 좋아요.

### 지표 선택 가이드

- **요약 품질 체크**: ROUGE
- **번역 품질 체크**: BLEU 또는 METEOR
- **의미 유사도가 중요할 때**: SemScore
- **종합 평가**: 네 가지 모두 + LLM-as-Judge

### 다음 노트북 예고

다음에는 RAG 시스템에서 가장 중요한 **Groundedness(근거성) 평가** — 답변이 컨텍스트에만 근거하는지, 즉 할루시네이션이 있는지 — 를 집중적으로 다뤄볼게요.